# AI Agent -- Built at SRM AP Workshop

This notebook deploys a complete AI agent with:
- Web search (DuckDuckGo)
- Calculator
- Beautiful chat UI (Gradio)
- Public URL anyone can access

**Steps:**
1. Run Cell 1 to install dependencies
2. Put your Groq API key in Cell 2
3. Run Cell 2
4. Click the public URL that appears
5. Share it with anyone

In [ ]:
# CELL 1: Install dependencies (run this first)
!pip install -q gradio groq duckduckgo-search

In [ ]:
# CELL 2: Full Agent + Gradio UI
# REPLACE THE API KEY BELOW WITH YOUR OWN FROM console.groq.com

import gradio as gr
from groq import Groq
from duckduckgo_search import DDGS

# ===== PUT YOUR KEY HERE =====
GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"
# =============================

client = Groq(api_key=GROQ_API_KEY)

# --- Tools ---
def calculator(expr):
    """Safe math evaluator."""
    allowed = set('0123456789+-*/.() ')
    if all(c in allowed for c in expr):
        try:
            return str(eval(expr))
        except:
            return "Error evaluating expression"
    return "Invalid expression"

def web_search(query):
    """Search the internet using DuckDuckGo."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
            return "\n".join([f"- {r['title']}: {r['body']}" for r in results])
    except Exception as e:
        return f"Search unavailable: {e}"

TOOLS = {
    "calculator": calculator,
    "web_search": web_search,
}

SYSTEM_PROMPT = """You are a helpful AI agent built at SRM AP Workshop.
You have access to these tools:
- calculator: Evaluate math expressions. Input: expression like '25*4+10'
- web_search: Search the internet for current information. Input: search query

To use a tool, write on its own line: ACTION: tool_name | input
You can use multiple tools across multiple steps.
After getting all results you need, provide your final answer.
If no tool is needed, just answer directly.

Be helpful, concise, and accurate."""

def agent(message, history):
    """Agent loop: ask LLM -> use tools if needed -> return answer."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": message}
    ]

    for step in range(5):  # max 5 tool calls
        response = client.chat.completions.create(
            model="qwen/qwen3.8-27b",
            messages=messages,
            max_tokens=600,
            temperature=0.3
        )
        reply = response.choices[0].message.content

        if "ACTION:" not in reply:
            return reply  # Final answer

        # Parse and execute tool call
        action_line = [l for l in reply.split('\n') if 'ACTION:' in l][0]
        parts = action_line.split("ACTION:")[1].strip().split("|", 1)
        tool_name = parts[0].strip()
        tool_input = parts[1].strip() if len(parts) > 1 else ""

        if tool_name in TOOLS:
            result = TOOLS[tool_name](tool_input)
        else:
            result = f"Unknown tool: {tool_name}"

        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user", "content": f"Tool result: {result}\nContinue or give your final answer."})

    return reply  # Return last response if max steps reached

# --- Launch Gradio ---
demo = gr.ChatInterface(
    fn=agent,
    title="AI Agent -- Built at SRM AP Workshop",
    description="I can search the web, do math, and answer your questions!",
    examples=[
        "What is 1847 * 293?",
        "What are the top AI agent frameworks in 2026?",
        "Compare React vs Next.js for freshers in India",
        "Calculate 15% tip on a bill of 2340 rupees",
    ]
)

demo.launch(share=True, debug=True)

# After running, look for:
# "Running on public URL: https://xxxxx.gradio.live"
# Click that URL or share it with anyone!